In [ ]:
# Fault classification model

import pandas as pd
import numpy as np
import mlflow
import mlflow.xgboost
from xgboost import XGBClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import accuracy_score, classification_report
import joblib
import os


print("Model traing pipeline Started succesfully!")

# Loading preprocessed data

if not os.path.exists('../data/raw_workshop_data.csv'):
    raise FileNotFoundError("Raw data file missing! Please run the preprocessing notebook first.")

df=pd.read_csv('../data/raw_workshop_data.csv')

# setting the mlflow name
mlflow.set_experiment("Vehivle Fault Classification")

#2. Text Vectorization (Feature Engineering)
# We need to transform the mechanical symptom text into numerical features using TF-IDF.
vectorizer = TfidfVectorizer(max_features=500)
X_text=vectorizer.fit_transform(df['symptom_text']).toarray()

# Combine text features with scaled numerical features (simulating manual matrix concat)
# For simplicity in this step, we'll train directly on the text features + basic engineering
from sklearn.preprocessing import LabelEncoder
label_encoder=LabelEncoder()
y=label_encoder.fit_transform(df['fault_category'])
    
#slpit in to train & test
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test= train_test_split(X_text, y, test_size=0.2, random_state=42)

# ### 3. Train XGBoost Model with Autologging active
# Start an MLflow training run
with mlflow.start_run(run_name="xgboost_base_line"):
    # Define Hyperparameters
    params={
        "objective": "multi:softprob",
        "num_class": len(np.unique(y)),
        "max_depth": 5,
        "learning rate": 0.1,
        "n_estimators": 100,
        "evel_metric": "mlogloss",
        "random_state": 42
    }

#log params to MLflow manually
mlflow.log_params(params)

# Initialize and train the model
print("Training XGbooost Classifier...")
model =XGBClassifier(params)
model.fit(X_train,y_train)

#Predict and Evaluate Performance

y_pred = model.predict(X_test)
accuracy = accuracy_score(y_test,y_pred)

#Log the Results to MLflow
mlflow.log_metric("accuracy", accuracy)

#Printing Local Reports
print(f"Training completed! Test accuracy: {accuracy*100:.2f}%")
print("/n Classification Report:\n", classification_report(y_test, y_pred, target_names=label_encoder.classes_))

#Save Supporting Files Locally
os.makedirs('../models', exist_ok=True)
joblib.dump(vectorizer, '../models/tfidf_vectorizer.pk1')

mlflow.xgboost.log_model(model, artifact_path="fault_classifier_model")
print(" Model and vectorizer successfully serialized and tracked in MLflow!")




        

Model traing pipeline Started succesfully!
Training XGbooost Classifier...


E:\Projects\vehicle-ai-platform\venv\Lib\site-packages\xgboost\core.py:748: FutureWarning: Pass `objective` as keyword args.
  warnings.warn(msg, FutureWarning)
2026/06/11 15:12:19 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


Training completed! Test accuracy: 100.00%
/n Classification Report:
                                precision    recall  f1-score   support

      Alternator / Drive Belt       1.00      1.00      1.00        37
        Fuel Injector Failure       1.00      1.00      1.00        31
      Glow Plug / Fuel System       1.00      1.00      1.00        53
Piston Ring / Valve Seal Wear       1.00      1.00      1.00        42
           Turbocharger Fault       1.00      1.00      1.00        37

                     accuracy                           1.00       200
                    macro avg       1.00      1.00      1.00       200
                 weighted avg       1.00      1.00      1.00       200



In [34]:
import mlflow
mlflow.end_run()
